# 第 2 课 · 任务、策略与采样

**本课用时约 30 分钟。** 拆开一道题里的任务、候选解题规则和策略抽样。

## 学完后你能做到

- 追踪一次策略如何挑选候选规则。
- 解释权重变化如何改变抽样概率。
- 说明固定随机种子解决了什么问题。

## 跟着操作

运行代码后，手动修改最后一段中的一个权重，重跑本节并比较概率。

运行代码格时，把光标放在格子里，按 **Shift + Enter**。看到输出后再运行下一格。若第一次误点或修改了上方变量，选 Jupyter 菜单里的 **Kernel → Restart Kernel and Run All Cells**，从第一格重新运行。本课不要求预先懂强化学习，也不需要 GPU。

## 任务与策略是两件事

题目的**任务**由「起点」和「指令」组成。候选规则是前进一格、两格、三格或后退一格。策略不直接存答案，而是给候选规则配权重，然后按权重比例抽取规则。

权重是正数也可以是整数计数。比如 `[5, 2, 1, 1]` 的总权重是 9，第一条规则被选中的概率是 5/9。采样引入随机性，同一策略有时会给出不同回答。

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
from rsi_lab import INITIAL, MOVES, NAMES, answer

task = 0
weights = INITIAL[task]
total = sum(weights)
print("题目:", NAMES[task])
for name, weight in zip(NAMES, weights):
    print(f"{name}: 权重 {weight}, 抽中概率 {weight / total:.1%}")
print("各规则在起点 8 的答案:", [answer(8, rule) for rule in range(len(MOVES))])

观察输出后，自己判断哪条规则最容易被抽到。在下一格里用固定随机种子抽 1,000 次。相同的种子和权重应该得到相同的计数。

In [ ]:
import random

rng = random.Random(19)
draws = rng.choices(range(len(weights)), weights=weights, k=1000)
counts = [draws.count(rule) for rule in range(len(weights))]
print(dict(zip(NAMES, counts)))
assert draws == random.Random(19).choices(range(len(weights)), weights=weights, k=1000)

现在改大一个候选规则的权重，观察抽样分布怎样改变。每次改完都重置种子，这样可把权重变化与随机序列变化区分开。

In [ ]:
changed = weights.copy()
changed[2] += 10
old_prob = weights[2] / sum(weights)
new_prob = changed[2] / sum(changed)
print(f"{NAMES[2]} 的抽样概率: {old_prob:.1%} -> {new_prob:.1%}")
new_draws = random.Random(19).choices(range(len(changed)), weights=changed, k=1000)
print("前进三格的实际次数:", counts[2], "->", new_draws.count(2))
assert new_prob > old_prob

## 自检

- [ ] 权重加总得到策略的归一化分母。
- [ ] 增加某条规则的权重会提高它的采样概率。
- [ ] 固定随机种子便于重复实验，但不会让模型更聪明。
- [ ] 我能指出哪个输入表示任务，哪个输入表示策略。

接下来会问一个重要问题：我们怎样知道抽到的答案是正确的？

[返回课程首页并选择下一节](recursive_self_improvement.ipynb)